# VN-Index Stochastic Calculus, Bayes-GARCH và MACD(6,26,12) Backtest

Notebook này trực quan hóa chiến lược Ito Bayes-GARCH và so sánh với MACD(6,26,12) trong điều kiện chỉ `long` hoặc `exit/cash`. Mục tiêu không phải là tạo khuyến nghị giao dịch, mà là đo lường khả năng dự báo, đặc tính rủi ro, hiệu quả lệnh và mức độ thay thế cho benchmark buy-and-hold trên VN-Index.

## 1. Khung Mô Hình

**Ito Bayes-GARCH** giả định biến động giá có nền tảng từ chuyển động Brown hình học:

`dS_t / S_t = mu_t dt + sigma_t dW_t`

Theo bổ đề Ito, log-price có động học:

`d log(S_t) = (mu_t - 0.5 sigma_t^2)dt + sigma_t dW_t`

Trong pipeline này, drift được ước lượng bằng Bayesian rolling mean, volatility được mô hình hóa bằng Student-t GARCH(1,1), và phần Bayes-GARCH được xấp xỉ bằng posterior quanh nghiệm MLE. Chiến lược vào lệnh khi expected log-return vượt một phần nhỏ forecast volatility.

**MACD(6,26,12)** là benchmark kỹ thuật sau tối ưu: `MACD = EMA6 - EMA26`, signal line là `EMA12(MACD)`. Chiến lược chỉ long khi MACD line lớn hơn signal line, ngược lại thoát vị thế. Tín hiệu được dịch 1 phiên để giảm look-ahead bias.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image, Markdown

OUTPUT_DIR = Path('outputs_stochastic_calculus')
metrics = pd.read_csv(OUTPUT_DIR / 'advanced_backtest_metrics_formatted.csv')
raw_metrics = pd.read_csv(OUTPUT_DIR / 'advanced_backtest_metrics.csv')
predictions = pd.read_csv(OUTPUT_DIR / 'predictions.csv', parse_dates=['date'])
ito_test = pd.read_csv(OUTPUT_DIR / 'strategy_backtest.csv', parse_dates=['date'])
macd_test = pd.read_csv(OUTPUT_DIR / 'macd_strategy_backtest.csv', parse_dates=['date'])

display(metrics.head(12))

## 2. Dữ Liệu và Dự Báo Train/Test

Tập dữ liệu được chia theo thời gian: train từ `2006-01-03` đến `2020-05-13`, test từ `2020-05-14` đến `2026-07-01`. Cách chia này phù hợp với bài toán chuỗi thời gian vì không trộn thông tin tương lai vào quá khứ.

Biểu đồ dưới đây cho thấy giá thực tế, one-step fit trên train, forecast trên test và dải bất định Brownian 90%.

![Forecast train test](outputs_stochastic_calculus/forecast_train_test.png)

## 3. Equity Curve: Ito Bayes-GARCH vs MACD vs Buy & Hold

Trên test, Buy & Hold có total return cao nhất, nhưng MACD có Sharpe cao hơn nhờ volatility thấp hơn. Ito Bayes-GARCH phòng thủ tốt về drawdown nhưng chưa tạo đủ upside trong giai đoạn test.

![Equity curve](outputs_stochastic_calculus/backtest_equity_curve.png)

## 4. Tín Hiệu Giao Dịch

Ito Bayes-GARCH tạo tín hiệu dựa trên expected drift so với forecast volatility. MACD tạo tín hiệu trend-following thuần kỹ thuật. Cả hai đều chỉ dùng long/exit, không short.

### Ito Bayes-GARCH signals

![Ito signals](outputs_stochastic_calculus/signals_on_price.png)

### MACD(6,26,12) signals

![MACD signals](outputs_stochastic_calculus/macd_signals_on_price.png)

## 5. MACD Indicator

MACD hoạt động như bộ lọc động lượng. Khi MACD line nằm trên signal line, thị trường được xem là có động lượng tăng đủ để giữ vị thế long. Khi MACD line cắt xuống dưới signal line, chiến lược thoát về cash.

![MACD indicator](outputs_stochastic_calculus/macd_indicator_test.png)

## 6. Đo Lường Rủi Ro Nâng Cao

Drawdown, rolling volatility và phân phối return là ba góc nhìn bổ sung cho chỉ số lợi nhuận. Một mô hình có return cao nhưng tail risk và drawdown quá lớn có thể kém hữu dụng hơn trong thực tế vận hành.

![Drawdown comparison](outputs_stochastic_calculus/drawdown_comparison_test.png)

![Rolling volatility](outputs_stochastic_calculus/rolling_volatility_comparison_test.png)

![Return distribution](outputs_stochastic_calculus/return_distribution_test.png)

In [ ]:
important_metrics = [
    'Market exposure',
    'Total VN-Index points',
    'Annual VN-Index points',
    'Total return',
    'CAGR',
    'Annualized mean return',
    'Annual volatility',
    'Sharpe ratio',
    'Sortino ratio',
    'Calmar ratio',
    'Max drawdown',
    'Daily VaR 95%',
    'Daily CVaR 95%',
    'Beta to buy-and-hold',
    'Annual alpha',
    'Trades',
    'Trade win rate',
    'Average trade return',
    'Average trade points',
]
display(metrics[metrics['Metric'].isin(important_metrics)].reset_index(drop=True))

## 7. Nhận Xét Học Thuật

1. **Về bản chất mô hình**: Ito Bayes-GARCH là mô hình hóa theo hướng xác suất liên tục, trong đó return được diễn giải qua drift và diffusion. Đây là cách tiếp cận có nền tảng mô hình hóa tốt vì volatility clustering được đưa vào qua GARCH, còn bất định dự báo được biểu diễn bằng dải Brownian/predictive band.

2. **Về khả năng dự báo**: directional accuracy trên test khoảng 55%, tức có tín hiệu hướng giá nhẹ nhưng chưa đủ mạnh để tự động vượt Buy & Hold về lợi nhuận. Điều này thường gặp với index: tín hiệu expected return nhỏ, trong khi biến động nhiễu lớn.

3. **Về MACD**: MACD không dự báo phân phối return, nhưng là bộ lọc động lượng đơn giản và khá hiệu quả trong giai đoạn test. MACD có Sharpe tốt hơn Buy & Hold vì giảm exposure và tránh một phần các đoạn giảm mạnh.

4. **Về rủi ro**: Ito có max drawdown test thấp hơn Buy & Hold, còn MACD có annual volatility và VaR/CVaR thấp hơn Buy & Hold. Điều này cho thấy cả hai chiến lược đều có vai trò risk filter, nhưng MACD tận dụng xu hướng tốt hơn trong giai đoạn test.

5. **Về alpha/beta**: beta của MACD thấp hơn Ito và thấp hơn nhiều so với benchmark, phản ánh mức tiếp xúc thị trường thấp. Annual alpha của MACD trên test dương, trong khi Ito âm, cho thấy MACD tạo active return tốt hơn trong giai đoạn kiểm định gần đây.

6. **Về thống kê lệnh**: MACD có số lệnh nhiều hơn, holding period ngắn hơn, trade win rate tốt hơn và average trade points cao hơn Ito trên test. Đây là dấu hiệu chiến lược MACD nhạy hơn với regime động lượng ngắn-trung hạn.

## 8. Hạn Chế và Hướng Mở Rộng

- Posterior Bayes-GARCH hiện là xấp xỉ quanh MLE, chưa phải MCMC đầy đủ.
- Nên kiểm định walk-forward refit để tránh phụ thuộc quá nhiều vào một lần fit cố định.
- Nên stress test transaction cost, threshold, MACD parameters và giai đoạn khủng hoảng.
- Có thể kết hợp hai mô hình: chỉ long khi cả drift Ito dương và MACD xác nhận trend, hoặc dùng volatility Bayes-GARCH để sizing vị thế cho MACD.